# Детектор разрядов SenuaLab на турецком наборе: годится ли как признак «эпилепсия да/нет»

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-20 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Проверить открытый детектор межприступных эпилептиформных разрядов (SenuaLab EEG-IED-Detection, обучен на vEpiSet) как признак уровня человека: доля окон с разрядами, максимум и среднее вероятности → ROC AUC «эпилепсия vs здоровые» |
| **Данные** | Turkish Epilepsy EEG (Kaggle `buraktaci/turkish-epilepsy`), 50 эпилепсия + 71 здоровый, первые 19 каналов (относительно уха), 500 Гц, куски по 15 с подряд |
| **Модели** | EEG-DINO Medium (лучшая у авторов, AUROC 0,936 на vEpiSet) и IEDNet-Lite (маленькая). Веса: HF `SenuaLab/EEG-IED-Detection`. **Без обучения**, только предсказание |
| **Вход модели** | 19 каналов в порядке Fp1, Fp2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T3, T4, T5, T6, Fz, Cz, Pz; окно 4 с; полоса 1–45 Гц → средний референт → передискретизация (EEG-DINO 200 Гц, IEDNet-Lite 250 Гц) → z-нормировка по всему окну (все каналы вместе) → обрезка ±8. Это код авторов (`preprocess_array`), берём его как есть |
| **Порог** | Подобран авторами на валидации vEpiSet по максимуму F1: EEG-DINO 0,619, IEDNet-Lite 0,600 |
| **Среда** | Kaggle, **процессор** (GPU не нужен), Internet On |
| **Журнал** | строка в `experiments_log.md`: добавить |

Куски по 15 с идут подряд (проверено в `h1_2026-10-18_zs_deepepilepsy-turkish`), поэтому склеиваем их в одну запись и режем окнами 4 с с шагом 2 с.

Для сравнения: в турецком наборе одна амплитуда сигнала даёт ROC AUC 0,79 (перекос набора), DeepEpilepsy после дообучения на NMT — 0,66 (`h1_2026-10-19_zs_deepepilepsy-nmt-to-turkish`).

## 1. Код и веса детектора

In [1]:
import os, sys, glob, time, json, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "braindecode==1.6.1"], check=True)
REPO = "/kaggle/working/EEG-IED-Detection"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SenuaLab/EEG-IED-Detection.git", REPO], check=True)
sys.path.insert(0, REPO)

import numpy as np, pandas as pd, torch, scipy.io
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from eeg_ied_detector.hub import resolve_checkpoint
from eeg_ied_detector.robust_train import CHANNELS_19, make_model, forward_model, preprocess_array
import braindecode
print("torch", torch.__version__, "| braindecode", braindecode.__version__, "| потоков", torch.get_num_threads())
DEV = torch.device("cpu")

MODELS = {}
for name in ["eegdino_medium", "iednet_lite"]:
    ck = torch.load(resolve_checkpoint(name), map_location="cpu", weights_only=True)
    m = make_model(ck["model_name"]); m.load_state_dict(ck["model_state_dict"]); m.eval()
    MODELS[name] = (m, ck)
    print(f"{name}: {ck['sfreq']} Гц, окно {ck['window_seconds']} с, порог {ck['validation_threshold_f1']:.3f}, "
          f"каналы в весах: {ck.get('channels')}")
print(CHANNELS_19)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.9/626.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.2 MB/s eta 0:00:00


Cloning into '/kaggle/working/EEG-IED-Detection'...
/usr/local/lib/python3.13/dist-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/usr/local/lib/python3.13/dist-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


torch 2.11.0+cpu | braindecode 1.6.1 | потоков 2


pytorch/eegdino_medium/model.pth: reconstructing file:   0%|          |  0.00B /  138MB            

pytorch/eegdino_medium/model.pth: downloading bytes:           |  0.00B            

eegdino_medium: 200 Гц, окно 4.0 с, порог 0.619, каналы в весах: ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']


pytorch/iednet_lite/model.pth: reconstructing file:   0%|          |  0.00B / 1.04MB            

pytorch/iednet_lite/model.pth: downloading bytes:           |  0.00B            

iednet_lite: 250 Гц, окно 4.0 с, порог 0.600, каналы в весах: ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']
['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']


## 2. Турецкий набор: подкачка и чтение (19 каналов в порядке детектора)

In [2]:
from concurrent.futures import ThreadPoolExecutor
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]

def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    for i, _ in enumerate(ex.map(warm, [f for f, _ in tfiles])):
        if (i + 1) % 20 == 0 or i + 1 == len(tfiles):
            print(f"подкачано {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)

# порядок в файле: FP1 FP2 F3 F4 FZ C3 C4 CZ P3 P4 PZ O1 O2 F7 F8 T3 T4 T5 T6 (индексы 0–18)
FILE_ORDER = ["Fp1", "Fp2", "F3", "F4", "Fz", "C3", "C4", "Cz", "P3", "P4", "Pz", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6"]
IDX = [FILE_ORDER.index(c) for c in CHANNELS_19]
print("индексы для порядка детектора:", IDX)
SF = 500

подкачано 20/121 10 с
подкачано 40/121 14 с
подкачано 60/121 17 с
подкачано 80/121 23 с
подкачано 100/121 23 с
подкачано 120/121 26 с
подкачано 121/121 26 с
индексы для порядка детектора: [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]


## 3. Предсказание по всем окнам (4 с, шаг 2 с) и признаки на человека

In [3]:
WIN, STEP, BS = 4 * SF, 2 * SF, 128

@torch.inference_mode()
def score(X):
    # X: (19, T) при 500 Гц -> вероятности разряда по окнам для каждой модели
    starts = np.arange(0, X.shape[1] - WIN + 1, STEP)
    out = {}
    for name, (m, ck) in MODELS.items():
        p = []
        for b in range(0, len(starts), BS):
            raw = np.stack([X[:, s:s + WIN] for s in starts[b:b + BS]])
            x = torch.from_numpy(preprocess_array(raw, source_sfreq=SF, target_sfreq=int(ck["sfreq"])))
            logits, _ = forward_model(m, x)
            p.append(logits.float().softmax(1)[:, 1].numpy())
        out[name] = np.concatenate(p)
    return out

rows, amp, PROBS = [], [], {}
t0 = time.time()
for i, (f, yy) in enumerate(tfiles):
    d = scipy.io.loadmat(f)["data"]
    X = np.concatenate([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])], axis=1).astype(np.float32)
    pr = score(X)
    PROBS[os.path.basename(f)] = pr
    r = {"file": os.path.basename(f), "y": yy, "minutes": X.shape[1] / SF / 60, "n_win": len(pr["eegdino_medium"]),
         "amp_std": float(np.median(X.std(1)))}   # амплитуда: медиана СКО по каналам (для сравнения с перекосом)
    for name, (m, ck) in MODELS.items():
        p, thr = pr[name], ck["validation_threshold_f1"]
        r[f"{name}_frac"] = float((p >= thr).mean())
        r[f"{name}_max"] = float(p.max())
        r[f"{name}_mean"] = float(p.mean())
        r[f"{name}_p95"] = float(np.percentile(p, 95))
    rows.append(r)
    if (i + 1) % 10 == 0 or i + 1 == len(tfiles):
        print(f"{i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/ied_scores_per_person.csv", index=False)
np.savez_compressed("/kaggle/working/ied_window_probs.npz", **{f"{k}|{n}": v for k, d_ in PROBS.items() for n, v in d_.items()})
print(df.groupby("y")[["minutes", "n_win"]].describe().T)

10/121 362 с
20/121 800 с
30/121 1107 с
40/121 1724 с
50/121 2174 с
60/121 2680 с
70/121 3108 с
80/121 3527 с
90/121 3906 с
100/121 4302 с
110/121 4692 с
120/121 5069 с
121/121 5109 с
y                        0            1
minutes count    71.000000    50.000000
        mean     20.742958    22.325000
        std       8.512278    14.410194
        min       8.500000     8.500000
        25%      17.250000    16.562500
        50%      18.750000    18.000000
        75%      20.500000    20.812500
        max      55.500000    67.000000
n_win   count    71.000000    50.000000
        mean    621.028169   668.540000
        std     255.364891   432.376405
        min     254.000000   254.000000
        25%     516.000000   495.750000
        50%     561.000000   539.000000
        75%     613.500000   623.250000
        max    1664.000000  2009.000000


## 4. ROC AUC «эпилепсия vs здоровые» с 95% ДИ (бутстреп по людям, 2000 повторов)

In [4]:
def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

y = df.y.to_numpy()
res = []
for name in MODELS:
    for feat, label in [("frac", "доля окон ≥ порога"), ("max", "максимум вероятности"), ("mean", "средняя вероятность"), ("p95", "95-й перцентиль")]:
        s = df[f"{name}_{feat}"].to_numpy()
        lo, hi = boot_ci(y, s)
        res.append({"модель": name, "признак": label, "ROC AUC": roc_auc_score(y, s), "ДИ_низ": lo, "ДИ_верх": hi,
                    "медиана здоровые": np.median(s[y == 0]), "медиана эпилепсия": np.median(s[y == 1]),
                    "Спирмен с амплитудой": spearmanr(s, df.amp_std).statistic})
s = df.amp_std.to_numpy(); lo, hi = boot_ci(y, s)
res.append({"модель": "—", "признак": "амплитуда (медиана СКО)", "ROC AUC": roc_auc_score(y, s), "ДИ_низ": lo, "ДИ_верх": hi,
            "медиана здоровые": np.median(s[y == 0]), "медиана эпилепсия": np.median(s[y == 1]), "Спирмен с амплитудой": 1.0})
res = pd.DataFrame(res)
res.to_csv("/kaggle/working/ied_auc.csv", index=False)
pd.set_option("display.width", 200)
print(res.round(3).to_string(index=False))

        модель                 признак  ROC AUC  ДИ_низ  ДИ_верх  медиана здоровые  медиана эпилепсия  Спирмен с амплитудой
eegdino_medium      доля окон ≥ порога    0.553   0.448    0.650             0.113              0.150                 0.175
eegdino_medium    максимум вероятности    0.433   0.332    0.538             0.990              0.988                 0.253
eegdino_medium     средняя вероятность    0.585   0.484    0.679             0.166              0.205                 0.192
eegdino_medium         95-й перцентиль    0.523   0.419    0.623             0.947              0.950                 0.241
   iednet_lite      доля окон ≥ порога    0.440   0.341    0.549             0.008              0.005                 0.265
   iednet_lite    максимум вероятности    0.464   0.362    0.570             0.977              0.951                 0.334
   iednet_lite     средняя вероятность    0.524   0.417    0.623             0.020              0.019                 0.377
   iedne

## 5. Сколько окон выше порога у здоровых (ложные срабатывания) и у больных

In [5]:
for name in MODELS:
    fr = df[f"{name}_frac"]
    print(f"{name}: доля окон ≥ порога — здоровые: медиана {fr[y==0].median():.3f}, среднее {fr[y==0].mean():.3f}; "
          f"эпилепсия: медиана {fr[y==1].median():.3f}, среднее {fr[y==1].mean():.3f}; "
          f"людей без единого срабатывания: здоровые {(fr[y==0]==0).sum()}/{(y==0).sum()}, эпилепсия {(fr[y==1]==0).sum()}/{(y==1).sum()}")
a = df[["eegdino_medium_mean", "iednet_lite_mean"]]
print("Спирмен между моделями (средняя вероятность):", round(spearmanr(a.iloc[:, 0], a.iloc[:, 1]).statistic, 3))

eegdino_medium: доля окон ≥ порога — здоровые: медиана 0.113, среднее 0.236; эпилепсия: медиана 0.150, среднее 0.248; людей без единого срабатывания: здоровые 0/71, эпилепсия 0/50
iednet_lite: доля окон ≥ порога — здоровые: медиана 0.008, среднее 0.021; эпилепсия: медиана 0.005, среднее 0.019; людей без единого срабатывания: здоровые 12/71, эпилепсия 12/50
Спирмен между моделями (средняя вероятность): 0.571


## 6. Выводы

Запуск 2026-10-20 на Kaggle, **процессор** (2 потока, GPU не тратился), предсказание ~85 мин. 121 человек, 77 520 окон по 4 с (шаг 2 с), записи 8,5–67 мин (медиана ~18 мин). Таблицы: `results/ied_senualab_turkish_auc.csv`, по людям — `results/ied_senualab_turkish_per_person.csv`.

| Признак на человека | EEG-DINO Medium: ROC AUC (95% ДИ) | IEDNet-Lite: ROC AUC (95% ДИ) |
|---|---|---|
| Доля окон ≥ порога (0,619 / 0,600) | 0,55 (0,45–0,65) | 0,44 (0,34–0,55) |
| Максимум вероятности | 0,43 (0,33–0,54) | 0,46 (0,36–0,57) |
| Средняя вероятность | 0,59 (0,48–0,68) | 0,52 (0,42–0,62) |
| 95-й перцентиль | 0,52 (0,42–0,62) | 0,58 (0,48–0,68) |
| *Для сравнения: амплитуда (медиана СКО по каналам)* | *0,70 (0,60–0,80)* | |

1. **Детектор технически работает** (веса грузятся, код авторов, вход 19 каналов 10-20, окно 4 с), но **больных от здоровых по нему не отличить**: все доверительные интервалы накрывают 0,5. Это хуже, чем дообученная на NMT DeepEpilepsy (0,66) и чем одна амплитуда (0,70 здесь; 0,79 по другой мере в блокноте REVE).
2. **Много ложных срабатываний у здоровых.** EEG-DINO отмечает ≥ порога в среднем 24 % окон у здоровых и 25 % у больных (медианы 11 % и 15 %); максимум вероятности ≈ 0,99 почти у всех. У авторов на vEpiSet ложных было ~0,6 в минуту (~4 % окон), здесь — в несколько раз больше: на чужой аппаратуре и при другом исходном референте (ухо) модель переносится плохо. IEDNet-Lite осторожнее (≈2 % окон), но тоже без разницы между группами; у 12 здоровых и 12 больных — ни одного срабатывания.
3. Модели между собой согласуются слабо (Спирмен средних вероятностей 0,57) и почти не связаны с амплитудой (Спирмен 0,18–0,40) — то есть перекос набора они не ловят, но и полезного сигнала не дают.
4. **Вывод для MVP:** в таком виде (без дообучения, с порогом авторов) детектор **не годится как второй признак** к модели «эпилепсия да/нет» на турецком наборе. Возможная польза — только как инструмент подсветки окон для врача после ручной проверки или после дообучения на размеченных разрядах (TUEV) — это отдельная задача.

**Ограничения:** метки турецкого набора — диагноз человека, а не разметка разрядов, поэтому точность самого детектора по окнам здесь не проверить; у многих больных межприступная ЭЭГ без разрядов, так что высокая оценка и не ожидалась; контроль — здоровые; качество записей и наличие артефактов не проверялись; порог взят у авторов, не подбирался. Метрики детектора на vEpiSet — только авторские.

**Лицензия весов:** код MIT; IEDNet-Lite и ResNet-Attention — CC BY 4.0 (указывать SenuaLab и vEpiSet); EEGPT и EEG-DINO — голова CC BY 4.0, кодировщики под условиями исходных проектов (BSD-3 / Apache-2.0), у EEG-DINO оговорка про возможные ограничения данных предобучения (важно для коммерции). Для исследовательского использования препятствий нет.